In [3]:
# Install all required libraries for retrieval and evaluation
!pip install pytrec_eval pyserini tqdm requests "pillow>=12.0" -q

# Verify installation
import pytrec_eval
print("pytrec_eval is successfully installed!")

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.7/159.7 MB 5.8 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 112.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 633.8/633.8 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 82.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.6 MB/s eta 0:00:00
ERR

In [4]:
import os
import requests
import json
import pandas as pd
import pytrec_eval
from tqdm import tqdm

# ==========================================
# 1. SETUP ENVIRONMENT & JAVA 21
# ==========================================
print("Setting up environment...")
!pip install pyserini pytrec_eval tqdm "pillow>=12.0" -q
!apt-get update
!apt-get install -y openjdk-21-jdk > /dev/null

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# ==========================================
# 2. DOWNLOAD & PREPARE INDIC-MARCO
# ==========================================
collection_url = "https://huggingface.co/datasets/saifulhaq9/indicmarco/resolve/main/hin_Deva/collection.tsv"
tsv_file = "indic_collection.tsv"
json_dir = "indic_jsonl"
os.makedirs(json_dir, exist_ok=True)

print("Downloading document collection...")
r = requests.get(collection_url, stream=True)
with open(tsv_file, 'wb') as f:
    for chunk in r.iter_content(chunk_size=8192):
        f.write(chunk)

print("Converting to JSONL format...")
with open(tsv_file, 'r', encoding='utf-8') as f_in, \
     open(os.path.join(json_dir, "docs.jsonl"), 'w', encoding='utf-8') as f_out:
    for line in tqdm(f_in):
        parts = line.strip().split('\t')
        if len(parts) >= 2:
            f_out.write(json.dumps({"id": str(parts[0]), "contents": parts[1]}, ensure_ascii=False) + "\n")

# Cleanup TSV to save disk space
os.remove(tsv_file)

# ==========================================
# 3. BUILD THE HINDI INDEX
# ==========================================
print("Building Lucene Index (Hindi)...")
!python -m pyserini.index.lucene \
  --collection JsonCollection \
  --input {json_dir} \
  --index indic_hindi_index \
  --generator DefaultLuceneDocumentGenerator \
  --threads 16 \
  --language hi \
  --storePositions --storeDocvectors --storeRaw

# Cleanup JSONL folder after indexing
import shutil
shutil.rmtree(json_dir)
# ==========================================
# 4. RUN BM25 RETRIEVAL
# ==========================================
# This uses your uploaded 'trec_dl19_hindi_query.tsv'
query_file = "/kaggle/input/datasets/patel1243/bm25-2019/trec_dl19_hindi_query.tsv"
output_run = "run.indic_bm25.txt"

print(f"Running retrieval for queries in {query_file}...")
!python -m pyserini.search.lucene \
  --index indic_hindi_index \
  --topics {query_file} \
  --output {output_run} \
  --bm25 \
  --language hi \
  --hits 1000 \
  --threads 16

# ==========================================
# 5. GENERATE RESULT MATRIX
# ==========================================
# This uses your uploaded 'pass_2019.qrels'
qrels_file = "/kaggle/input/datasets/patel1243/bm25-2019/pass_2019.qrels"

print("Calculating Evaluation Matrix...")
with open(qrels_file, 'r') as f_qrel:
    qrel = pytrec_eval.parse_qrel(f_qrel)
with open(output_run, 'r') as f_run:
    run = pytrec_eval.parse_run(f_run)

# Define requested cuts: 1, 10, 100, 1000
metrics = {
    'ndcg_cut_1', 'ndcg_cut_10', 'ndcg_cut_100', 'ndcg_cut_1000',
    'recall_1', 'recall_10', 'recall_100', 'recall_1000'
}

evaluator = pytrec_eval.RelevanceEvaluator(qrel, metrics)
results = evaluator.evaluate(run)

# Aggregate averages
avg_results = {m: pytrec_eval.compute_aggregated_measure(m, [results[q][m] for q in results]) for m in metrics}

print("\n" + "="*50)
print("       INDIC-MARCO HINDI EVALUATION MATRIX")
print("="*50)

# Sort metrics by type and depth
sorted_metrics = sorted(avg_results.keys(), key=lambda x: (x.split('_')[0], int(x.split('_')[-1])))
for m in sorted_metrics:
    print(f"{m:25}: {avg_results[m]:.4f}")


Setting up environment...
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:5 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease    
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease                        
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,829 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [357 B]      
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,302 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted a

8841823it [02:16, 64613.72it/s]


Building Lucene Index (Hindi)...
2026-03-17 07:23:50,711 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:208) - Setting log level to INFO
2026-03-17 07:23:50,714 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:211) - ============ Loading Index Configuration ============
2026-03-17 07:23:50,714 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:212) - AbstractIndexer settings:
2026-03-17 07:23:50,725 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:213) -  + DocumentCollection path: indic_jsonl
2026-03-17 07:23:50,725 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:214) -  + CollectionClass: JsonCollection
2026-03-17 07:23:50,726 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:215) -  + Index path: indic_hindi_index
2026-03-17 07:23:50,726 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:216) -  + Threads: 16
2026-03-17 07:23:50,727 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:217) -  + Optimize (merge segments)? fals

In [6]:
# ==========================================
# 4. RUN BM25 RETRIEVAL
# ==========================================
# This uses your uploaded 'trec_dl19_hindi_query.tsv'
query_file = "/kaggle/input/datasets/patel1243/bm25-2020/trec_dl20_hindi_query.tsv"
output_run = "run.indic_bm25.txt"

print(f"Running retrieval for queries in {query_file}...")
!python -m pyserini.search.lucene \
  --index indic_hindi_index \
  --topics {query_file} \
  --output {output_run} \
  --bm25 \
  --language hi \
  --hits 1000 \
  --threads 16

# ==========================================
# 5. GENERATE RESULT MATRIX
# ==========================================
# This uses your uploaded 'pass_2019.qrels'
qrels_file = "/kaggle/input/datasets/patel1243/bm25-2020/pass_2020.qrels"

print("Calculating Evaluation Matrix...")
with open(qrels_file, 'r') as f_qrel:
    qrel = pytrec_eval.parse_qrel(f_qrel)
with open(output_run, 'r') as f_run:
    run = pytrec_eval.parse_run(f_run)

# Define requested cuts: 1, 10, 100, 1000
metrics = {
    'ndcg_cut_1', 'ndcg_cut_10', 'ndcg_cut_100', 'ndcg_cut_1000',
    'recall_1', 'recall_10', 'recall_100', 'recall_1000'
}

evaluator = pytrec_eval.RelevanceEvaluator(qrel, metrics)
results = evaluator.evaluate(run)

# Aggregate averages
avg_results = {m: pytrec_eval.compute_aggregated_measure(m, [results[q][m] for q in results]) for m in metrics}

print("\n" + "="*50)
print("       INDIC-MARCO HINDI EVALUATION MATRIX")
print("="*50)

# Sort metrics by type and depth
sorted_metrics = sorted(avg_results.keys(), key=lambda x: (x.split('_')[0], int(x.split('_')[-1])))
for m in sorted_metrics:
    print(f"{m:25}: {avg_results[m]:.4f}")

Running retrieval for queries in /kaggle/input/datasets/patel1243/bm25-2020/trec_dl20_hindi_query.tsv...
Mar 17, 2026 7:57:07 AM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false
Running /kaggle/input/datasets/patel1243/bm25-2020/trec_dl20_hindi_query.tsv topics, saving to run.indic_bm25.txt...
100%|███████████████████████████████████████████| 54/54 [00:06<00:00,  8.65it/s]
Calculating Evaluation Matrix...

       INDIC-MARCO HINDI EVALUATION MATRIX
ndcg_cut_1               : 0.2870
ndcg_cut_10              : 0.2383
ndcg_cut_100             : 0.2145
ndcg_cut_1000            : 0.2782
recall_1                 : 0.0170
recall_10                : 0.0789
recall_100               : 0.2118
recall_1000              : 0.3717
